# RAG Question-Answering Pipeline

**Local VS Code Edition — ChromaDB + Vertex AI Embeddings + Gemini 2.5 Pro**

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline that
answers questions grounded in a corpus of PDF research papers. It was originally
authored on Google Colab against Vertex AI Matching Engine and has been ported
for local execution using a persistent ChromaDB vector store.

## Architecture

| Stage | Component | Purpose |
|-------|-----------|---------|
| Document source | Google Cloud Storage (GCS) | Holds the source PDFs |
| Loader | `PyPDFLoader` (LangChain) | Extracts page-level text from each PDF |
| Splitter | `RecursiveCharacterTextSplitter` | Chunks text into overlapping windows |
| Embeddings | Vertex AI `text-embedding-005` | Encodes chunks into 768-dim vectors |
| Vector store | **ChromaDB (local, persistent)** | Indexes vectors for similarity search |
| Retriever | LangChain similarity retriever | Top-k chunk lookup for a query |
| LLM | **Gemini 2.5 Pro** via `ChatGoogleGenerativeAI` | Generates the grounded answer |
| Orchestration | LangChain retrieval chain | Stitches retriever + LLM with a strict prompt |

## Notebook structure

1. **Installs** — single consolidated cell
2. **Imports** — single consolidated cell
3. **Configuration** — project, bucket, paths, hyperparameters
4. **Phase 1** — Vertex AI client initialization
5. **Phase 2** — Document download from GCS, parsing, and chunking
6. **Phase 5** — ChromaDB vector store creation
7. **Phase 6** — Batched embedding and ingestion (with retry on quota errors)
8. **Phase 7** — Sanity-check similarity search
9. **Phase 8** — LLM, retriever, prompt template, RAG chain, and `ask()` function
10. **Phase 9** — End-to-end question-answering examples

> Phases 3 (index/endpoint creation), 4 (embeddings bucket creation), and 10
> (print Matching Engine IDs) from the original Colab version have been
> intentionally removed because they were specific to Vertex AI Matching
> Engine, which has been replaced by ChromaDB.


## Authentication (One-Time Setup)

Authentication is **handled outside this notebook** — we no longer use
`google.colab.auth.authenticate_user()` because we are running locally in
VS Code.

Before launching the notebook, run the following **once** in your terminal:

```bash
# Authenticate your user identity (opens a browser)
gcloud auth application-default login

# Pin the active project so client libraries use it by default
gcloud config set project gill-adta5770-rag
```

This writes Application Default Credentials (ADC) to
`~/.config/gcloud/application_default_credentials.json` (Linux/macOS) or
`%APPDATA%\gcloud\application_default_credentials.json` (Windows). Every
Google client library used in this notebook — `google.cloud.storage`,
`google.cloud.aiplatform`, `vertexai`, `langchain_google_vertexai`,
`langchain_google_genai` — will pick up those credentials automatically.

**Required IAM roles** on your user account for the target project:

- `roles/storage.objectViewer` — to read the source PDFs from the GCS bucket
- `roles/aiplatform.user` — to call Vertex AI embeddings and Gemini

If you ever see `google.auth.exceptions.DefaultCredentialsError`, re-run
`gcloud auth application-default login`.


## 1. Installs

A single consolidated install cell. Run this once per environment, then
restart the Jupyter kernel before continuing.


In [ ]:
# Consolidated dependency installation.
# Use --quiet to keep notebook output readable during pip resolution.

# Google Cloud SDKs (auth + GCS + Vertex AI + Gemini)
%pip install --quiet --upgrade google-cloud-aiplatform google-cloud-storage vertexai google-genai

# LangChain core stack
%pip install --quiet --upgrade langchain langchain-core langchain-classic langchain-community langchain-text-splitters

# LangChain Google integrations (Vertex AI embeddings, Gemini chat model)
%pip install --quiet --upgrade langchain-google-community langchain-google-vertexai langchain-google-genai

# Local vector store (replaces Vertex AI Matching Engine)
%pip install --quiet --upgrade chromadb langchain-chroma

# PDF parsing (PyPDFLoader backend — no OCR / poppler / tesseract needed
# because we download each PDF locally before parsing)
%pip install --quiet --upgrade pypdf

print("Installation complete. Please restart the kernel before running the next cell.")


## 2. Imports

All imports are consolidated into this single cell so that dependency usage is
transparent at a glance and ordering issues cannot arise between phases.


In [ ]:
# --- Standard library ---
import os
import time
import textwrap

# --- Google Cloud / Vertex AI ---
from google.cloud import storage                          # GCS bucket access
from google.cloud import aiplatform                       # Vertex AI SDK (init only)
import vertexai                                           # Vertex AI SDK (init only)
from google.api_core.exceptions import ResourceExhausted  # For quota / rate-limit retry

# --- LangChain core (prompts, debug toggle) ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.globals import set_verbose

# --- LangChain document loading and chunking ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- LangChain RAG chain primitives ---
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# --- Vertex AI embedding model wrapper ---
from langchain_google_vertexai import VertexAIEmbeddings

# --- Gemini chat model wrapper ---
from langchain_google_genai import ChatGoogleGenerativeAI

# --- ChromaDB vector store (local, persistent — replaces Vertex AI Matching Engine) ---
from langchain_chroma import Chroma

# Quick version sanity check — useful for reproducibility in the report.
import langchain
from langchain_core import __version__ as langchain_core_version
from langchain_classic import __version__ as langchain_classic_version
from langchain_community import __version__ as langchain_community_version
print(f"aiplatform SDK version       : {aiplatform.__version__}")
print(f"Vertex AI SDK version        : {vertexai.__version__}")
print(f"LangChain version            : {langchain.__version__}")
print(f"LangChain Core version       : {langchain_core_version}")
print(f"LangChain Classic version    : {langchain_classic_version}")
print(f"LangChain Community version  : {langchain_community_version}")


## 3. Configuration

All tunable parameters live here so the rest of the notebook can be re-read
without scrolling back for project IDs, paths, or hyperparameters.


In [ ]:
# --- Google Cloud project + region ---
PROJECT_ID = 'gill-adta5770-rag'   # GCP project hosting the bucket and Vertex AI quotas
REGION     = 'us-central1'         # Region for Vertex AI (must support text-embedding-005 and Gemini)

# --- Source PDF location in Google Cloud Storage ---
BUCKET_NAME   = 'domain-docs'         # GCS bucket containing the corpus
folder_prefix = 'documents/pdfs/'     # Folder (object key prefix) inside the bucket
BUCKET_URI    = f"gs://{BUCKET_NAME}/{folder_prefix}"  # Full gs:// URI for logging only

# --- Local working directories (created at runtime if they don't exist) ---
LOCAL_PDF_DIR        = './temp_pdfs'    # Where GCS PDFs are downloaded for parsing
CHROMA_PERSIST_DIR   = './chroma_db'    # Where ChromaDB stores its on-disk index
CHROMA_COLLECTION    = 'rag_qa_collection'  # Logical name for the Chroma collection

# --- Embedding model (Vertex AI) ---
EMBEDDING_MODEL_NAME = 'text-embedding-005'  # 768-dim Google text embedding model
DIMENSIONS           = 768                   # Output dimensionality of text-embedding-005

# --- Retrieval hyperparameters ---
NUMBER_OF_RESULTS         = 10   # Top-k chunks to retrieve for each query
SEARCH_DISTANCE_THRESHOLD = 0.6  # Reserved for future similarity-score filtering

# --- Ingestion hyperparameters ---
CHUNK_SIZE     = 400   # Characters per chunk (small chunks = more precise retrieval)
CHUNK_OVERLAP  = 50    # Overlap between adjacent chunks (preserves boundary context)
BATCH_SIZE     = 25    # Chunks per embedding API call
MAX_RETRIES    = 5     # Retries on Vertex AI quota errors (exponential backoff)

# --- LLM (generation) ---
LLM_MODEL_NAME = 'gemini-2.5-pro'  # Reasoning model used to compose the final answer

print(f"Project       : {PROJECT_ID}")
print(f"Region        : {REGION}")
print(f"Source bucket : {BUCKET_URI}")
print(f"Chroma store  : {CHROMA_PERSIST_DIR}")


## Phase 1 — Vertex AI Client Initialization

Although the **vector store** runs locally (ChromaDB), we still call Vertex AI
for two things:

1. **Embedding generation** via `text-embedding-005`
2. **Answer generation** via Gemini 2.5 Pro

Both client SDKs (`aiplatform` and `vertexai`) need to be initialized once
with the project ID and region so subsequent calls know where to route
requests. We also instantiate a `storage.Client` to verify that
Application Default Credentials are working before we hit the network in
Phase 2.

> The original Colab notebook used Phase 1 to **delete leftover Matching
> Engine indexes and endpoints** to stop billing. That cleanup is no longer
> needed — local Chroma has nothing to clean up.


In [ ]:
# Initialize the Vertex AI SDKs. These calls only configure the client;
# they do NOT make any network requests on their own.
aiplatform.init(project=PROJECT_ID, location=REGION)
vertexai.init(project=PROJECT_ID, location=REGION)

# Instantiate a GCS client. This will fail fast if Application Default
# Credentials aren't configured — useful to surface auth issues early
# rather than mid-pipeline.
storage_client = storage.Client(project=PROJECT_ID)
print(f"Authenticated with project: {storage_client.project}")


## Phase 2 — Document Loading & Chunking

This phase performs three things:

1. **List** the PDFs that live under `gs://{BUCKET_NAME}/{folder_prefix}`.
2. **Download** each PDF locally into `./temp_pdfs/` and parse it with
   `PyPDFLoader`. We download first because parsing a local file is
   dramatically faster than streaming through `GCSFileLoader`, and it
   avoids invoking the heavy `unstructured` / OCR path.
3. **Chunk** every page into overlapping windows using
   `RecursiveCharacterTextSplitter`. Each chunk inherits page metadata and
   is tagged with its source path, document filename, and a sequential
   chunk index — these tags are later used for retrieval filtering and
   citation display in the `ask()` output.


### 2a. List the source PDFs

In [ ]:
# Print every blob under the configured prefix so it's clear which files
# will be processed downstream. This is a read-only listing call.
bucket = storage_client.bucket(BUCKET_NAME)
print(f"Listing PDFs in {BUCKET_URI}\n")

for blob in bucket.list_blobs(prefix=folder_prefix):
    print(f"  - {blob.name}")


### 2b. Download and parse each PDF

In [ ]:
# Make sure the local download folder exists.
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)

print(f"Processing documents from {BUCKET_URI}\n")

# Accumulator for parsed page-level Documents from every PDF.
all_documents = []

# Stream blobs under the prefix. Each blob is one object in GCS.
for blob in bucket.list_blobs(prefix=folder_prefix):
    # Skip "directory" placeholder blobs and non-PDF objects.
    if blob.name.endswith("/") or not blob.name.lower().endswith(".pdf"):
        continue

    print(f"  Loading document: {blob.name}")

    # 1. Download the PDF locally first (fast — avoids the heavy OCR path
    #    that GCSFileLoader triggers via the `unstructured` library).
    document_name = blob.name.split("/")[-1]
    local_file_path = os.path.join(LOCAL_PDF_DIR, document_name)
    blob.download_to_filename(local_file_path)

    # 2. Parse with PyPDFLoader — produces one LangChain Document per page.
    loader = PyPDFLoader(local_file_path)
    documents_from_blob = loader.load()

    # 3. Stamp every page-document with provenance metadata. These fields
    #    are surfaced in the citation block of the `ask()` formatter and
    #    are also used for filtered retrieval in Phase 9.
    doc_source_prefix = f"gs://{BUCKET_NAME}"
    doc_source_suffix = "/".join(blob.name.split("/")[0:-1])
    source = f"{doc_source_prefix}/{doc_source_suffix}"

    for document in documents_from_blob:
        document.metadata["source"] = source                # gs://bucket/prefix
        document.metadata["document_name"] = document_name  # original PDF filename
        all_documents.append(document)

print(f"\n# of document pages loaded (pre-chunking) = {len(all_documents)}")


### 2c. Split pages into overlapping chunks

Smaller chunks (400 chars with 50-char overlap) give the retriever more
fine-grained units to match against. The overlap prevents key facts from
being cut at chunk boundaries.


In [ ]:
# RecursiveCharacterTextSplitter walks down a list of separators
# (\n\n, \n, " ", "") until each piece fits within `chunk_size`.
# This produces semantically cleaner chunks than naive fixed-size slicing.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    is_separator_regex=False,
)

# Apply the splitter to the page-level Documents from Phase 2b.
doc_splits = text_splitter.split_documents(all_documents)

# Tag each chunk with a stable sequential index. Useful for debugging
# retrieval results and for downstream citation rendering.
for idx, split in enumerate(doc_splits):
    split.metadata["chunk"] = idx

print(f"# of chunks (post-splitting) = {len(doc_splits)}")


## Phase 5 — Create the Local ChromaDB Vector Store

> **Note.** Phases 3 and 4 from the original Colab version have been removed
> because they configured Vertex AI Matching Engine (cloud index creation,
> endpoint deployment, and a dedicated GCS embeddings bucket). None of that
> infrastructure is needed for a local vector store, so phase numbering jumps
> from 2 directly to 5 to preserve the original phase labels for the parts
> that remain.

ChromaDB is an open-source vector database that runs entirely in-process
and persists to disk under `CHROMA_PERSIST_DIR`. We pair it with the same
Vertex AI embedding model the original notebook used (`text-embedding-005`)
so that the **embedding semantics are identical** to the cloud version —
only the storage and similarity-search backend has changed.

If `CHROMA_PERSIST_DIR` already exists and contains a previous run's
collection, Chroma will reopen it; otherwise it will create a new one.


In [ ]:
# Vertex AI embedding model. Authenticated via Application Default Credentials.
# Output is a 768-dim float vector per input string.
embedding_model = VertexAIEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    project=PROJECT_ID,
)

# Open (or create) a persistent local Chroma collection. The `embedding_function`
# is invoked automatically on every `add_texts` and on every query — we never
# have to pass raw vectors ourselves.
vsvectordb = Chroma(
    collection_name=CHROMA_COLLECTION,
    embedding_function=embedding_model,
    persist_directory=CHROMA_PERSIST_DIR,
)

print(f"ChromaDB collection '{CHROMA_COLLECTION}' ready at: {CHROMA_PERSIST_DIR}")
print(f"Existing items in collection: {vsvectordb._collection.count()}")


## Phase 6 — Embed and Ingest Chunks

Each chunk is sent to Vertex AI for embedding, then written to ChromaDB
along with its metadata. We batch requests (`BATCH_SIZE = 25`) to stay
under the embedding API's tokens-per-minute quota, and we wrap each call in
an exponential-backoff retry loop so a transient `ResourceExhausted` does
not abort the whole ingest.

Re-running this cell will append duplicates. If you want a clean slate,
delete the `./chroma_db` directory and re-run Phase 5.


In [ ]:
# Pull texts and metadatas out of the Document objects so we can batch them.
texts = [doc.page_content for doc in doc_splits]
metadatas = [doc.metadata for doc in doc_splits]

total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

# Iterate over fixed-size slices to keep request size bounded.
for i in range(0, len(texts), BATCH_SIZE):
    batch_texts     = texts[i : i + BATCH_SIZE]
    batch_metadatas = metadatas[i : i + BATCH_SIZE]
    batch_num       = i // BATCH_SIZE + 1

    print(f"Adding batch {batch_num}/{total_batches} (size: {len(batch_texts)})...")

    retries = 0
    while retries < MAX_RETRIES:
        try:
            # Embeds the batch via Vertex AI AND inserts into Chroma in one call.
            vsvectordb.add_texts(texts=batch_texts, metadatas=batch_metadatas)
            break  # Success — exit the retry loop.

        except ResourceExhausted:
            # Quota / rate-limit error. Back off exponentially (10s, 20s, 40s, ...).
            retries += 1
            sleep_time = (2 ** retries) * 5
            print(f"   [!] Quota hit. Sleeping {sleep_time}s before retry "
                  f"(attempt {retries}/{MAX_RETRIES})...")
            time.sleep(sleep_time)

        except Exception as e:
            # Anything else: log and bail on this batch so a single bad chunk
            # cannot freeze the whole ingest.
            print(f"   [!] Unexpected error on batch {batch_num}: {e}")
            break

    # Light pacing between batches to smooth the request rate.
    time.sleep(2)

print(f"\nIngestion complete. Collection now has "
      f"{vsvectordb._collection.count()} items.")


## Phase 7 — Sanity-Check the Vector Store

Before wiring up the LLM, confirm that semantic search is returning
reasonable matches for a known concept from the corpus. If this returns
empty results or unrelated chunks, debug Phase 6 before moving on.


In [ ]:
# k=2 keeps the output short for inspection; we'll bump k for real queries.
sanity_results = vsvectordb.similarity_search("What is epidemiology?", k=2)

for i, hit in enumerate(sanity_results, start=1):
    print(f"--- Hit {i} ---")
    print(f"document_name: {hit.metadata.get('document_name')}")
    print(f"chunk        : {hit.metadata.get('chunk')}")
    print(f"content      : {hit.page_content[:300]}...")
    print()


## Phase 8 — Build the RAG Chain and the `ask()` Function

This is the heart of the notebook. We assemble:

1. **The LLM** — Gemini 2.5 Pro with conservative decoding parameters
   (`temperature=0.2`) for factual answers.
2. **The retriever** — a similarity retriever over the Chroma collection,
   returning the top `NUMBER_OF_RESULTS` chunks.
3. **The prompt template** — instructs the model to answer **only** from
   the retrieved context and to refuse otherwise.
4. **The retrieval chain** — LangChain's `create_retrieval_chain` glues
   the retriever and the document-stuffing prompt together.
5. **`ask()`** — a thin wrapper that lets us pass per-query overrides
   (`k`, `filters`) and pretty-prints the citations alongside the answer.


### 8a. Instantiate the Gemini LLM

In [ ]:
# Gemini 2.5 Pro via the LangChain wrapper. The wrapper authenticates
# through Application Default Credentials (the same `gcloud auth ADC` you ran
# at the start), and routes through the configured project + region.
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL_NAME,
    project=PROJECT_ID,
    location=REGION,
    max_output_tokens=8192,  # Plenty of room for grounded answers + reasoning
    temperature=0.2,         # Low temperature → more deterministic, factual replies
    top_p=0.8,
    top_k=40,
    verbose=True,
)


### 8b. Configure the retriever

In [ ]:
# `as_retriever` exposes the Chroma collection through the standard LangChain
# retriever interface. We start with no filter; `ask()` will inject one
# at query time when the caller supplies `filters=...`.
retriever = vsvectordb.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": NUMBER_OF_RESULTS,
    },
)


### 8c. Define the answer prompt

The prompt explicitly forbids the model from answering outside the
retrieved context. This is what turns a generic chat model into a
grounded, citation-anchored QA system.


In [ ]:
# Strict prompt: the model must say "I cannot determine..." rather than
# inventing facts. The {input} and {context} placeholders are filled in by
# the retrieval chain at invocation time.
template = """SYSTEM: You are an intelligent assistant helping the users with their questions on research papers.

Question: {input}

Strictly use ONLY the following pieces of context to answer the question at the end. Think step-by-step and then answer.

Do not try to make up an answer:
  - If the answer to the question cannot be determined from the context alone, say "I cannot determine the answer to that."
  - If the context is empty, just say "I do not know the answer to that."

=============
{context}
=============

Question: {input}
Helpful Answer:"""

qa_prompt = ChatPromptTemplate.from_template(template)


### 8d. Wire the chain together

In [ ]:
# `create_stuff_documents_chain` builds a chain that "stuffs" all retrieved
# documents into the {context} slot of the prompt and asks the LLM for an answer.
combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)

# `create_retrieval_chain` then prepends the retriever step so that calling
# `.invoke({"input": question})` will: (1) embed the query, (2) fetch top-k
# chunks from Chroma, (3) stuff them into the prompt, (4) call Gemini.
retrieval_qa_chain = create_retrieval_chain(retriever, combine_docs_chain)

# Turn on LangChain's verbose mode so chain steps are logged inline —
# extremely useful when demoing the pipeline in a class presentation.
set_verbose(True)


### 8e. Pretty-printing helpers

In [ ]:
def wrap(s):
    """Wrap long strings to 120 columns for readable cell output."""
    return "\n".join(textwrap.wrap(s, width=120, break_long_words=False))


def formatter(result):
    """Render a retrieval-chain result as a citation-style block:
    the original query, every retrieved chunk with its provenance metadata,
    then the final grounded answer."""
    print(f"Query: {result['input']}")
    print("." * 80)

    # Walk through every retrieved chunk and emit its metadata + content.
    if "context" in result.keys():
        for idx, ref in enumerate(result["context"]):
            print("-" * 80)
            print(f"REFERENCE #{idx}")
            print("-" * 80)
            if "score" in ref.metadata:
                print(f"Matching Score: {ref.metadata['score']}")
            if "source" in ref.metadata:
                print(f"Document Source: {ref.metadata['source']}")
            if "document_name" in ref.metadata:
                print(f"Document Name: {ref.metadata['document_name']}")
            print("." * 80)
            print(f"Content: \n{wrap(ref.page_content)}")

    print("." * 80)
    print(f"Response: {wrap(result['answer'])}")
    print("." * 80)


### 8f. The `ask()` function

Public entry point for end-to-end question answering. Accepts an optional
`filters` argument that mirrors the original Colab API:

```python
filters = {
    "namespace": "document_name",
    "allow_list": ["Some-Paper.pdf", "Another-Paper.pdf"],
}
```

Internally we translate that into Chroma's native `where` filter syntax
(`{field: value}` for a single value, `{field: {"$in": [...]}}` for a list)
so callers don't need to know which vector store is underneath.


In [ ]:
def ask(query, k=NUMBER_OF_RESULTS, filters=None):
    """Run a RAG query end-to-end and pretty-print the result.

    Parameters
    ----------
    query : str
        The natural-language question.
    k : int
        Number of chunks to retrieve (defaults to NUMBER_OF_RESULTS).
    filters : dict or None
        Optional metadata filter, e.g.
        ``{"namespace": "document_name", "allow_list": ["foo.pdf"]}``.
        Translated to a Chroma ``where`` clause internally.
    """
    # Per-query override of top-k.
    retriever.search_kwargs["k"] = k

    if filters:
        # Translate the original Matching-Engine-style filter into Chroma's
        # native `where` syntax.
        namespace = filters["namespace"]
        allow_list = filters["allow_list"]

        if len(allow_list) == 1:
            # Single allowed value → simple equality filter.
            chroma_filter = {namespace: allow_list[0]}
        else:
            # Multiple allowed values → use $in operator.
            chroma_filter = {namespace: {"$in": allow_list}}

        retriever.search_kwargs["filter"] = chroma_filter
    else:
        # Make sure we don't carry a stale filter over from a previous call.
        retriever.search_kwargs.pop("filter", None)

    # Invoke the chain: embed → retrieve → stuff → generate.
    result = retrieval_qa_chain.invoke({"input": query})

    # Render the result as a citation block.
    return formatter(result)


### 8g. Quick end-to-end sanity check

In [ ]:
# Simplest possible end-to-end smoke test for the full RAG chain.
ask("What does epidemiology mean?")


## Phase 9 — Question-Answering Examples

Demonstrates two usage patterns:

1. **Filtered retrieval** — restrict the search to a specific document,
   useful when the user already knows which paper holds the answer.
2. **Open retrieval** — let the retriever pick from the entire corpus.

Each call below prints the retrieved chunks (with their source filenames)
and Gemini's grounded answer.


### 9a. Filtered query — restrict to a single document

In [ ]:
# Restrict retrieval to a single PDF by exact filename match on the
# `document_name` metadata field stamped during Phase 2.
filters = {
    "namespace": "document_name",
    "allow_list": ["Infectious-disease-epidemiology.pdf"],
}

ask("What does epidemiology mean?", filters=filters)


### 9b. Open queries — search across the entire corpus

In [ ]:
ask("What food source was linked to the 2024 large-scale norovirus outbreak across multiple schools in a Korean city?")


In [ ]:
ask("What novel clade or sub-lineage of the monkeypox virus was associated with the 2024 outbreak in Kamituga, South Kivu province?")


In [ ]:
ask("According to the ESPAUR report for 2024 to 2025, what are the key findings regarding antimicrobial utilisation and resistance?")


In [ ]:
ask("What are the interim estimates for the effectiveness of the COVID-19 vaccine during the 2024-2025 season?")


---

**End of pipeline.** The notebook intentionally stops here — the original
Phase 10 only printed Vertex AI Matching Engine resource IDs, which no
longer exist in the local ChromaDB version.
